# Extract x1dbk1, x1dtrace, and x1dbk2 from test-data

This notebook builds on `inspiration/extraction_script.py` and focuses on the test data in `test-data/`. It will:

- create `x1d` files if they do not exist
- extract `x1dbk1`, `x1dtrace`, and `x1dbk2` using the same offsets/sizes logic

The trace selection is interactive using `%matplotlib widget` and `ipywidgets`, so the plot should appear inline.

Run the cells top to bottom.

In [ ]:
%matplotlib widget

import os
from astropy.io import fits

from airglow import extraction_utils as extract_utils

# Optional: set CRDS paths if they are not already configured.
# Update CRDS_PATH to match your local cache location if needed.
CRDS_PATH = "/Users/parke/crds_cache"
if "CRDS_PATH" not in os.environ:
    os.environ["CRDS_PATH"] = CRDS_PATH
    os.environ.setdefault("CRDS_SERVER_URL", "https://hst-crds.stsci.edu")
    os.environ.setdefault("iref", f"{CRDS_PATH}/references/hst/iref/")
    os.environ.setdefault("jref", f"{CRDS_PATH}/references/hst/jref/")
    os.environ.setdefault("oref", f"{CRDS_PATH}/references/hst/oref/")
    os.environ.setdefault("lref", f"{CRDS_PATH}/references/hst/lref/")
    os.environ.setdefault("nref", f"{CRDS_PATH}/references/hst/nref/")
    os.environ.setdefault("uref", f"{CRDS_PATH}/references/hst/uref/")

In [ ]:
TEST_DATA_DIR = extract_utils.find_test_data_dir()
flt_files = sorted(TEST_DATA_DIR.glob("*_flt.fits"))

print(f"Test data dir: {TEST_DATA_DIR}")
print("FLT files:")
for f in flt_files:
    print(f"- {f.name}")

In [ ]:
# Pick a single file at a time.
TARGET_FILE = TEST_DATA_DIR / "of9b05010_flt.fits"
print(f"Target: {TARGET_FILE.name}")

In [ ]:
# Display the interactive plot and click to select the trace.
# Use the buttons below the plot to use the default or clear the selection.
fig, ax = extract_utils.setup_trace_selector(TARGET_FILE)

In [ ]:
# Run the extractions using the selected trace location.
trace_y = (
    extract_utils.SELECTED_TRACE_Y
    if extract_utils.SELECTED_TRACE_Y is not None
    else extract_utils.DEFAULT_TRACE_Y
)
if trace_y is None:
    raise ValueError("No trace selected and no default trace available.")
else:
    print(f"Using trace y: {trace_y}")

extract_utils.extract_standard_x1d(
    TARGET_FILE,
    overwrite=True,
    manual_traceloc=trace_y,
)

print(f"Using trace y: {trace_y}")
extract_utils.extract_background_traces(
    TARGET_FILE,
    overwrite=True,
    manual_traceloc=trace_y,
)

In [ ]:
print("\nOutputs:")
for suffix in ("x1dbk1", "x1dtrace", "x1dbk2"):
    out = TARGET_FILE.with_name(TARGET_FILE.name.replace("_flt", f"_{suffix}"))
    print(f"- {out.name} ({'exists' if out.exists() else 'missing'})")
    if out.exists():
        a2center = fits.getdata(out, 1)["a2center"]
        print(f"  a2center: {a2center}")

In [ ]:
# Plot the extraction locations for the generated x1d files on the FLT image.
fig, ax = extract_utils.plot_extraction_locations(TARGET_FILE)

In [ ]:
from matplotlib import pyplot as plt
from astropy.io import fits

root = TARGET_FILE.name.split("_")[0]
x1d_files = sorted(TEST_DATA_DIR.glob(f"{root}*x1d*.fits"))

plt.figure()
for file in x1d_files:
    h = fits.open(file)
    w, f = [h[1].data[k][0] for k in ("WAVELENGTH", "FLUX")]
    plt.plot(w, f, label=file.name)
plt.legend()